In [3]:
pip install implicit

   ---------------------------------------- 0.0/750.8 kB ? eta -:--:--
    --------------------------------------- 10.2/750.8 kB ? eta -:--:--
    --------------------------------------- 10.2/750.8 kB ? eta -:--:--
   - ------------------------------------- 30.7/750.8 kB 217.9 kB/s eta 0:00:04
   -- ------------------------------------ 41.0/750.8 kB 178.6 kB/s eta 0:00:04
   --- ----------------------------------- 61.4/750.8 kB 252.2 kB/s eta 0:00:03
   ---- ---------------------------------- 92.2/750.8 kB 327.7 kB/s eta 0:00:03
   ------ ------------------------------- 122.9/750.8 kB 379.3 kB/s eta 0:00:02
   ---------- --------------------------- 204.8/750.8 kB 541.9 kB/s eta 0:00:02
   ------------ ------------------------- 256.0/750.8 kB 605.3 kB/s eta 0:00:01
   ----------------- -------------------- 337.9/750.8 kB 723.4 kB/s eta 0:00:01
   -------------------------- ------------- 501.8/750.8 kB 1.0 MB/s eta 0:00:01
   ------------------------------- -------- 593.9/750.8 kB 1.1 MB

In [33]:
# If not already installed:
# %pip install -q implicit

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, csc_matrix, save_npz, load_npz
import implicit
import os, json, time, platform, sys


In [35]:
# Columns in your dataset
USER_COL = "user"
ITEM_COL = "isbn"
RATING_COL = "rating"

# Pruning thresholds (tune if needed)
MIN_USER_RATINGS = 5
MIN_ITEM_RATINGS = 5

# Implicit conversion: treat "positive" feedback as rating >= threshold
IMPLICIT_THRESHOLD = 7  # adjust to your rating scale

# ALS hyperparams (we'll tune later)
ALS_FACTORS = 128
ALS_REG = 0.05
ALS_ITERS = 20
ALS_USE_CG = True
ALS_ALPHA = 40.0      # confidence strength (we’ll sweep later too)
USE_BM25 = True       # weighting before building confidence (helps long-tail)

TOPK_EVAL = 10  # Recall@K/MAP@K cutoff

# Load your prepared csv
ratings_prepared_data = pd.read_csv("data/df_ratings_prepared.csv")
len(ratings_prepared_data), ratings_prepared_data.head(3)


(1009521,
      user        isbn  rating
 0  276725  034545104X       0
 1    2313  034545104X       5
 2    6543  034545104X       0)

In [36]:
def prune_and_map_implicit(df, user_col, item_col, rating_col,
                           min_user_ratings=5, min_item_ratings=5,
                           implicit_threshold=7):
    x = df[[user_col, item_col, rating_col]].copy()
    x[user_col] = x[user_col].astype(str)
    x[item_col] = x[item_col].astype(str)
    x[rating_col] = pd.to_numeric(x[rating_col], errors="coerce")
    x = x.dropna(subset=[rating_col])

    # Convert to implicit positives
    x["implicit"] = (x[rating_col] >= implicit_threshold).astype(np.float32)
    x = x[x["implicit"] > 0]

    # Iterative prune (users/items with too few positives)
    def _pass(y):
        uc = y.groupby(user_col)[item_col].count()
        ic = y.groupby(item_col)[user_col].count()
        keep_u = uc[uc >= min_user_ratings].index
        keep_i = ic[ic >= min_item_ratings].index
        return y[y[user_col].isin(keep_u) & y[item_col].isin(keep_i)]

    last = -1
    while last != len(x):
        last = len(x)
        x = _pass(x)

    ucat = pd.Categorical(x[user_col])
    icat = pd.Categorical(x[item_col])
    x["uid"] = ucat.codes.astype(np.int32)
    x["iid"] = icat.codes.astype(np.int32)

    uid2raw = pd.Series(ucat.categories, index=np.arange(len(ucat.categories)))
    iid2raw = pd.Series(icat.categories, index=np.arange(len(icat.categories)))
    return x[["uid","iid","implicit"]].rename(columns={"implicit":"val"}), uid2raw, iid2raw

df_imp, uid2raw, iid2raw = prune_and_map_implicit(
    ratings_prepared_data, USER_COL, ITEM_COL, RATING_COL,
    MIN_USER_RATINGS, MIN_ITEM_RATINGS, IMPLICIT_THRESHOLD
)

n_users = int(df_imp["uid"].max()) + 1
n_items = int(df_imp["iid"].max()) + 1
n_users, n_items, len(df_imp)


(4726, 6124, 74052)

In [38]:
def train_test_split_per_user(df, holdout_per_user=1, seed=123):
    rng = np.random.default_rng(seed)
    x = df.copy()
    x["_rand"] = rng.random(len(x))
    x = x.sort_values(["uid","_rand"])
    x["row_idx"] = x.groupby("uid").cumcount()
    sizes = x.groupby("uid")["row_idx"].transform("max") + 1
    x["is_test"] = x["row_idx"] >= (sizes - holdout_per_user)
    train = x[~x["is_test"]][["uid","iid","val"]]
    test  = x[ x["is_test"]][["uid","iid","val"]]
    return train, test

train_df, test_df = train_test_split_per_user(df_imp, holdout_per_user=1, seed=123)
len(train_df), len(test_df)


(69326, 4726)

In [39]:
def build_user_item(train_df, n_users, n_items):
    rows = train_df["uid"].to_numpy()
    cols = train_df["iid"].to_numpy()
    vals = train_df["val"].astype(np.float32).to_numpy()
    R = csr_matrix((vals, (rows, cols)), shape=(n_users, n_items), dtype=np.float32)  # users×items
    return R

R_ui = build_user_item(train_df, n_users, n_items)  # implicit positives (0/1)

# Optional weighting to reduce popularity bias
if USE_BM25:
    R_weighted = implicit.nearest_neighbours.bm25_weight(R_ui.T).T  # still users×items
else:
    # TF-IDF alternative:
    # R_weighted = implicit.nearest_neighbours.tfidf_weight(R_ui.T).T
    R_weighted = R_ui

# Build confidence matrix C = alpha * R_weighted (library expects ITEM×USER)
C_iu = (R_weighted * ALS_ALPHA).T.tocsr()  # items×users
C_iu.shape


(6124, 4726)

In [40]:
def train_als(C_item_user, factors=128, reg=0.05, iterations=20, use_cg=True, seed=42):
    model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=reg,
        iterations=iterations,
        use_cg=use_cg,
        calculate_training_loss=False,
        random_state=seed,
    )
    model.fit(C_item_user, show_progress=False)
    return model, model.user_factors.copy(), model.item_factors.copy()  # U (users), V (items)

als_model, U, V = train_als(C_iu, factors=ALS_FACTORS, reg=ALS_REG, iterations=ALS_ITERS, use_cg=ALS_USE_CG)
U.shape, V.shape


((6124, 128), (4726, 128))

In [49]:
# === FIX: force R_train to match the fitted model's shapes ===
import numpy as np
from scipy.sparse import csr_matrix, vstack, hstack

# 1) Start from whatever you currently have (R_weighted or C_iu.T)
#    If you still have C_iu from training, it's best to base on that:
if 'C_iu' in globals():
    # items×users -> users×items
    R_train = C_iu.T.tocsr()
else:
    # fallback: use the weighted user×item matrix you built before training
    R_train = R_weighted.tocsr()

# 2) Get target shapes from the model (this is the source of truth)
N_USERS = int(als_model.user_factors.shape[0])
N_ITEMS = int(als_model.item_factors.shape[0])

# 3) Coerce R_train to (N_USERS, N_ITEMS) by padding/truncating
rows, cols = R_train.shape

# pad/truncate rows (users)
if rows < N_USERS:
    R_train = vstack([R_train, csr_matrix((N_USERS - rows, cols), dtype=R_train.dtype)], format='csr')
elif rows > N_USERS:
    R_train = R_train[:N_USERS, :]

# pad/truncate cols (items)
rows, cols = R_train.shape
if cols < N_ITEMS:
    R_train = hstack([R_train, csr_matrix((rows, N_ITEMS - cols), dtype=R_train.dtype)], format='csr')
elif cols > N_ITEMS:
    R_train = R_train[:, :N_ITEMS]

# 4) Rebuild 'seen', popularity, and a couple of guards
seen = [set(R_train[uid].indices.tolist()) for uid in range(N_USERS)]
item_pop = np.asarray(R_train.getnnz(axis=0)).astype(np.int32)
pop_rank = np.argsort(-item_pop)

# (optional) sanity prints
print("R_train shape (coerced):", R_train.shape)
print("Model factors shapes   :", als_model.user_factors.shape, als_model.item_factors.shape)


R_train shape (coerced): (6124, 4726)
Model factors shapes   : (6124, 128) (4726, 128)


In [51]:
def als_recommend_for_user(uid, topn=10, blend_pop=0.0):
    uid = int(np.asarray(uid).item())
    if uid < 0 or uid >= R_train.shape[0] or R_train[uid].nnz == 0:
        return []

    user_items_row = R_train[uid]

    # sanitize filter items to [0, N_ITEMS)
    if seen[uid]:
        filt = np.fromiter((i for i in seen[uid] if 0 <= i < N_ITEMS), dtype=np.int32)
    else:
        filt = np.empty(0, dtype=np.int32)

    item_ids, scores = als_model.recommend(
        userid=uid,
        user_items=user_items_row,            # 1×N_ITEMS
        N=max(topn*2, topn),
        filter_items=filt,
        filter_already_liked_items=False,
        recalculate_user=False
    )

    if blend_pop <= 0:
        return [(int(i), float(s)) for i, s in zip(item_ids[:topn], scores[:topn])]

    # optional blend with popularity
    s = np.asarray(scores, dtype=np.float32)
    p = item_pop[item_ids].astype(np.float32)

    def mm(x):
        lo, hi = float(x.min()), float(x.max())
        return np.zeros_like(x) if hi <= lo else (x - lo) / (hi - lo)

    final = (1 - blend_pop) * mm(s) + blend_pop * mm(p)
    order = np.argsort(-final)[:topn]
    return [(int(item_ids[i]), float(final[i])) for i in order]


def evaluate_als(gt_dict, topn=10, blend_pop=0.0):
    recalls, maps = [], []
    for uid in gt_dict.index:
        uid_int = int(np.asarray(uid).item())
        if uid_int < 0 or uid_int >= R_train.shape[0] or R_train[uid_int].nnz == 0:
            continue
        preds = [i for i,_ in als_recommend_for_user(uid_int, topn=topn, blend_pop=blend_pop)]
        truth = gt_dict[uid_int]
        recalls.append(recall_at_k(preds, truth, k=topn))
        maps.append(apk(preds, truth, k=topn))
    return (float(np.mean(recalls)) if recalls else 0.0,
            float(np.mean(maps))    if maps    else 0.0)


In [53]:
# Ground truth from TEST (items the user actually interacted with)
gt = test_df.groupby("uid")["iid"].apply(set)

# Drop cold users that vanished after pruning/splitting
covered = set(np.where(R_train.getnnz(axis=1) > 0)[0].tolist())
gt = gt[[u in covered for u in gt.index]]

def recall_at_k(pred, truth, k=10):
    if not truth: return 0.0
    return len(set(pred[:k]) & truth) / len(truth)

def apk(pred, truth, k=10):
    if not truth: return 0.0
    score = 0.0
    hits = 0
    for i, p in enumerate(pred[:k], 1):
        if p in truth:
            hits += 1
            score += hits / i
    return score / min(len(truth), k)

def evaluate_als(gt_dict, topn=10, blend_pop=0.0):
    recalls, maps = [], []
    for uid in gt_dict.index:
        preds = [i for i,_ in als_recommend_for_user(uid, topn=topn, blend_pop=blend_pop)]
        truth = gt_dict[uid]
        recalls.append(recall_at_k(preds, truth, k=topn))
        maps.append(apk(preds, truth, k=topn))
    return float(np.mean(recalls)), float(np.mean(maps))

rec10, map10 = evaluate_als(gt, topn=TOPK_EVAL, blend_pop=0.0)
rec10, map10


(0.0016985138004246285, 0.00030372729417315404)

In [55]:
grid = {
    "factors": [64, 128, 256],
    "reg":     [0.01, 0.05, 0.1],
    "alpha":   [20.0, 40.0, 80.0],
    "iters":   [15, 20],
}

results = []
for k in grid["factors"]:
    for reg in grid["reg"]:
        for a in grid["alpha"]:
            for it in grid["iters"]:
                C_try = (R_weighted * a).T.tocsr()
                model, U_, V_ = train_als(C_try, factors=k, reg=reg, iterations=it, use_cg=True)
                als_model = model  # bind for recommend()
                r, m = evaluate_als(gt, topn=TOPK_EVAL, blend_pop=0.0)
                results.append((k, reg, a, it, r, m))
                print(f"f={k:>3} reg={reg:.3f} alpha={a:>4} it={it:>2} -> R@{TOPK_EVAL}={r:.4f}  MAP@{TOPK_EVAL}={m:.4f}")

res_als = pd.DataFrame(results, columns=["factors","reg","alpha","iters","Recall@10","MAP@10"])
res_als.sort_values(["Recall@10","MAP@10"], ascending=[False, False]).head(10)


f= 64 reg=0.010 alpha=20.0 it=15 -> R@10=0.0015  MAP@10=0.0002
f= 64 reg=0.010 alpha=20.0 it=20 -> R@10=0.0013  MAP@10=0.0002
f= 64 reg=0.010 alpha=40.0 it=15 -> R@10=0.0019  MAP@10=0.0003
f= 64 reg=0.010 alpha=40.0 it=20 -> R@10=0.0015  MAP@10=0.0002
f= 64 reg=0.010 alpha=80.0 it=15 -> R@10=0.0019  MAP@10=0.0003
f= 64 reg=0.010 alpha=80.0 it=20 -> R@10=0.0011  MAP@10=0.0002
f= 64 reg=0.050 alpha=20.0 it=15 -> R@10=0.0013  MAP@10=0.0002
f= 64 reg=0.050 alpha=20.0 it=20 -> R@10=0.0013  MAP@10=0.0002
f= 64 reg=0.050 alpha=40.0 it=15 -> R@10=0.0015  MAP@10=0.0002
f= 64 reg=0.050 alpha=40.0 it=20 -> R@10=0.0015  MAP@10=0.0002
f= 64 reg=0.050 alpha=80.0 it=15 -> R@10=0.0019  MAP@10=0.0004
f= 64 reg=0.050 alpha=80.0 it=20 -> R@10=0.0019  MAP@10=0.0003
f= 64 reg=0.100 alpha=20.0 it=15 -> R@10=0.0015  MAP@10=0.0002
f= 64 reg=0.100 alpha=20.0 it=20 -> R@10=0.0013  MAP@10=0.0002
f= 64 reg=0.100 alpha=40.0 it=15 -> R@10=0.0015  MAP@10=0.0002
f= 64 reg=0.100 alpha=40.0 it=20 -> R@10=0.0013  MAP@10

,factors,reg,alpha,iters,Recall@10,MAP@10
10,64,0.05,80.0,15,0.001911,0.000403
4,64,0.01,80.0,15,0.001911,0.000323
16,64,0.10,80.0,15,0.001911,0.000293
11,64,0.05,80.0,20,0.001911,0.000277
2,64,0.01,40.0,15,0.001911,0.000274
34,128,0.10,80.0,15,0.001699,0.000331
27,128,0.05,40.0,20,0.001699,0.000304
17,64,0.10,80.0,20,0.001699,0.000251
29,128,0.05,80.0,20,0.001486,0.000300
26,128,0.05,40.0,15,0.001486,0.000273


In [ ]:
MODEL_DIR = "models/als_v1"
os.makedirs(MODEL_DIR, exist_ok=True)

# Factors
np.save(os.path.join(MODEL_DIR, "U_user_factors.npy"), U.astype(np.float32))
np.save(os.path.join(MODEL_DIR, "V_item_factors.npy"), V.astype(np.float32))

# ID maps
uid2raw.to_frame("user_raw").to_csv(os.path.join(MODEL_DIR, "uid2raw.csv"), index_label="uid")
iid2raw.to_frame("isbn_raw").to_csv(os.path.join(MODEL_DIR, "iid2raw.csv"), index_label="iid")

# Persist weighted train matrix if you want fast “seen” reconstruction
save_npz(os.path.join(MODEL_DIR, "R_train_weighted.npz"), R_train.astype(np.float32))

# Metadata
meta = {
    "saved_at_unix": time.time(),
    "python": sys.version,
    "platform": platform.platform(),
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "implicit": implicit.__version__},
    "n_users": int(U.shape[0]),
    "n_items": int(V.shape[0]),
    "params": {
        "ALS_FACTORS": ALS_FACTORS,
        "ALS_REG": ALS_REG,
        "ALS_ITERS": ALS_ITERS,
        "ALS_ALPHA": ALS_ALPHA,
        "ALS_USE_CG": ALS_USE_CG,
        "USE_BM25": USE_BM25,
        "TOPK_EVAL": TOPK_EVAL,
    }
}
with open(os.path.join(MODEL_DIR, "metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("✅ Saved ALS model to:", MODEL_DIR)


In [ ]:
LOAD_DIR = "models/als_v1"

U_loaded = np.load(os.path.join(LOAD_DIR, "U_user_factors.npy"))
V_loaded = np.load(os.path.join(LOAD_DIR, "V_item_factors.npy"))
uid2raw_loaded = pd.read_csv(os.path.join(LOAD_DIR, "uid2raw.csv"), index_col="uid")["user_raw"]
iid2raw_loaded = pd.read_csv(os.path.join(LOAD_DIR, "iid2raw.csv"), index_col="iid")["isbn_raw"]
R_train_loaded = load_npz(os.path.join(LOAD_DIR, "R_train_weighted.npz"))

with open(os.path.join(LOAD_DIR, "metadata.json"), "r") as f:
    meta_loaded = json.load(f)

# Rebuild a minimal implicit model shell for fast .recommend()
als_loaded = implicit.als.AlternatingLeastSquares(
    factors=U_loaded.shape[1],
    regularization=meta_loaded["params"]["ALS_REG"],
    iterations=1,  # not training
    use_cg=meta_loaded["params"]["ALS_USE_CG"],
    random_state=42,
)
als_loaded.user_factors = U_loaded.copy()
als_loaded.item_factors = V_loaded.copy()

# Seen items and popularity
seen_loaded = [set(R_train_loaded[uid].indices.tolist()) for uid in range(R_train_loaded.shape[0])]
item_pop_loaded = np.asarray(R_train_loaded.getnnz(axis=0)).astype(np.int32)

def als_loaded_recommend(uid, topn=10):
    filt = np.array(list(seen_loaded[uid]), dtype=np.int32)
    ids, scores = als_loaded.recommend(
        userid=uid, user_items=R_train_loaded, N=topn, filter_items=filt, recalculate_user=False
    )
    return [(int(i), float(s)) for i, s in zip(ids, scores)]

# Example
uid = 0
rec_items = als_loaded_recommend(uid, topn=10)
[iid2raw_loaded[i] for i,_ in rec_items]
